# Lesson 11 — Simplex Revealed

## Learning objectives

1. Walk through one full simplex iteration (entering variable, ratio test, pivot).
2. Understand the revised simplex method (basis inverse, factorizations).
3. Recognize and handle degeneracy and cycling.
4. State the worst-case (Klee-Minty) and average-case complexity.

## 1. The dictionary view

Express each basic variable as a function of the non-basic variables:

$$x_B = B^{-1} b - B^{-1} N x_N, \quad z = c_B^\top B^{-1} b + (c_N^\top - c_B^\top B^{-1} N) x_N.$$

The reduced-cost vector is $\bar c_N = c_N - c_B^\top B^{-1} N$. Optimality: $\bar c_N \ge 0$.

## 2. One iteration

1. **Pricing:** find $j$ with $\bar c_j < 0$ (Dantzig: most negative; Bland: smallest index {cite:p}`Bland1977`).
2. **Ratio test:** find $i$ minimizing $\bar b_i / d_i$ over rows with $d_i > 0$, where $\bar b = B^{-1} b$ is the *current* dictionary RHS and $d = B^{-1} A_j$. (If $d \le 0$ everywhere: unbounded.)
3. **Pivot:** swap $A_j$ into the basis, evict the row-$i$ basic variable.

Updating $B^{-1}$ explicitly is O($m^2$); modern solvers maintain LU factors and update them incrementally (Forrest-Tomlin, Bartels-Golub) {cite:p}`Bertsimas1997`.

## 3. Revised simplex

Don't compute $B^{-1}$ explicitly. Solve $B^\top y = c_B$ for the dual; price with $\bar c_j = c_j - A_j^\top y$. Solve $B d = A_j$ for the column update. Two solves per iteration with a sparse LU.

Why it matters: an LP with $m = 10^5$ rows and $10\%$-sparse columns has a basis matrix with maybe $10^6$ nonzeros — $B$ is sparse, $B^{-1}$ is not.

## 4. Degeneracy and cycling

A degenerate pivot makes zero progress. A *cycle* visits the same basis twice. Anti-cycling rules:

- **Bland's rule:** entering = smallest index with $\bar c_j < 0$; leaving = smallest index in tie. Provably finite {cite:p}`Bland1977`.
- **Lexicographic perturbation:** treat ties using a lexicographic ordering of rows.

In practice, modern LP solvers use the **Harris two-pass ratio test** to allow slightly larger pivots and improve **numerical stability** — note this is a stability device, *not* an anti-cycling guarantee. Anti-cycling proper still rests on Bland's rule or lexicographic perturbation.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np

def revised_simplex(c, A, b, max_iter=200):
    m, n = A.shape
    # Phase 2 only: assume b >= 0, identity columns at the end gave a starting basis.
    # (Real implementations use Phase 1; left as exercise.)
    A = np.hstack([A, np.eye(m)])
    c = np.concatenate([c, np.zeros(m)])
    basis = list(range(n, n + m))

    for it in range(max_iter):
        B = A[:, basis]
        cB = c[basis]
        y = np.linalg.solve(B.T, cB)
        rc = c - A.T @ y
        nb = [j for j in range(n + m) if j not in basis]
        rc_nb = rc[nb]
        if np.all(rc_nb >= -1e-9): return basis, c[basis] @ np.linalg.solve(B, b)
        j_in = nb[int(np.argmin(rc_nb))]
        d = np.linalg.solve(B, A[:, j_in])
        if np.all(d <= 1e-9): raise RuntimeError("unbounded")
        ratios = np.where(d > 1e-9, np.linalg.solve(B, b) / d, np.inf)
        i_out = int(np.argmin(ratios))
        basis[i_out] = j_in
    raise RuntimeError("max iter")

c = np.array([-3, -2.0])
A = np.array([[1, 1], [2, 1], [1, 3.0]])
b = np.array([4, 5, 6.0])
basis, z = revised_simplex(c, A, b)
print("optimum (max):", -z, "basis:", basis)


## 5. Complexity

- Worst case: exponential. Klee-Minty {cite:p}`KleeMinty1972` constructed an LP where Dantzig's rule visits all $2^n$ vertices.
- Average case: polynomial. Borgwardt (1982) proved expected polynomial behaviour under random LP distributions; Spielman & Teng's smoothed-analysis result (2004) strengthened this further. *(See {cite:p}`Bertsimas1997` for both citations within an LP textbook context.)*

In practice: LP solvers are blazingly fast, and the choice of simplex vs IPM (Lesson 12) is more about problem structure than worst-case theory.

## References
{cite:p}`Bertsimas1997` Ch 3, {cite:p}`Dantzig1963`, {cite:p}`Bland1977`.